# Suite REF — References: one format for objects, relation entries, frames and parameters

A reference designates what a request or a response points at ([CAP-130: references to objects, entries and frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-130.md#decided)):
`References.OfObject`, an object, by its path (a singleton, an owned object or a
schema object), never a counted one, which nothing names (object identity 40 and 55 in [CAP-118: singletons, owned and counted objects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-118.md#decided));
`References.OfEntry`, a relation entry, by its relation's path and all its links and properties; `References.OfFrame`,
a frame, each symbol's object and each variable's value, and what it witnesses, objects' property values
(`References.OfProperties`) and relation entries, each by its objects' references, and the errors annotated on it
(`References.OfError`, [CAP-309: errors annotated on frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-309)). `References.of(store, x)`
gives one, a value of the store's, written and read as any value is. A reference is resolved within a transaction, as
every read of a store's data is ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md)): a map whose symbol is given an object's reference
binds what it designates, and gives no frame where the store holds none; a frame's reference is resolved by a map
whose symbols are given the references it binds, and an entry's is matched by a pattern over the objects it links. `References.OfParameter`, a parameter, holds its name and
nothing else; a binding in the scope that encloses it resolves it, not a store. A parameter reference may stand in the place of anything with a schema
([CAP-312: a parameter reference in place of anything with a schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-312.md#decided)): it is read through `Visitors.OfAny.as_parameter`, every other
visitor reads its place as empty, and the plain form writes it marked with its schema where the reader could take it for
a value of the schema expected.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store are `Proxy` objects typed
loosely, so callbacks are annotated `(t: any)`; a frame is a `Map`, plain mappings are `Map`s, ints `bigint`s.

In [ ]:
import { Builders as B, Changes, Comparison as C, JSON as Json, Paths, Plain, Proxies, Reachable, References, Transactions,
  Validators, YAML } from "@mbse/schemas/Framework";
import { DecodeError, LookupError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, same } from "./support.js";

/** crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by key; crm.Note, counted,
 * about a person. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject,
      Note: S.OfObject, About: S.OfRelation })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]).properties({
        since: S.OfInt }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
      ]).adjacencies({ trio: (t: any) => t.relation([S.OfRelation((r: any) => r.links([ // no name of its own
        (t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]))]).me("a") }),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({
        key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }).adjacencies({
        about: (t: any) => t.relation([_.About]).me("note") }),
      (_: any) => _.About.name("crm.About").links([(t: any) => t.name("note"), (t: any) => t.name("person")]),
    ])]))]);
}

const store: any = new B.Store();
store.transaction(crm);
const [Person, Book, Note] = ["crm.Person", "crm.Book", "crm.Note"].map((name) => store.registered(name));
const frames = store.transaction((t: any) => t.uid("people").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
  .symbols({ book: Book, ada: Person, bob: Person, cy: Person }).spec([
    (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada), bob: (e: any) => e.contact(_.bob),
      cy: (e: any) => e.contact(_.cy) }),
    (_: any) => _.ada.name("Ada").knows([(e: any) => e.known(_.bob).since(2020n), (e: any) => e.known(_.cy).since(2021n)]),
    (_: any) => _.bob.name("Bob"),
    (_: any) => _.cy.name("Cy"),
  ])]))]));
const [book, ada, bob, cy] = ["book", "ada", "bob", "cy"].map((name) => frames[0].get(name)); // each its reference: a frame holds no visitor

/** The contact the book holds by `key`, within the transaction open: found again by navigating from the singleton. */
function contact(key: string): any {
  return store.singleton("crm.TheBook").contacts({ key });
}

/** A note about each person, upserted with the key `notes/1` for the frame that reaches it. */
function notes(t: any): any {
  return t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ contact: Person })])
    .then([(n: any) => n.upsert((u: any) => u
    .key("notes/1").exists([(p: any) => p.symbols({ contact: Person, note: Note }).spec([
      (_: any) => _.note.text("hi").about([_.contact])])]))]))]);
}

function plain(reference: any): any {
  return Plain.ToPlain(store)(reference.as_schema(), reference);
}

/** Frames given back, each symbol's reference in its plain form, a schema object as itself, a counted object, which the
 * frame refuses, as "counted": what tells frames apart, a frame holding references made afresh in each transaction
 * ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md)). */
function plains(frames: any[]): Map<string, unknown>[] {
  const one = (frame: any, name: string): unknown => {
    let held: any;
    try {
      held = frame.get(name);
    } catch (error) {
      if (error instanceof LookupError) return "counted";
      throw error;
    }
    return store.name_of(held.as_schema()) === "References.OfObject" ? plain(held) : held;
  };
  return frames.map((frame) => new Map([...frame.keys()].map((name: string) => [name, one(frame, name)])));
}

/** A path's plain form: from a singleton through entries, each [i, o, keys]. */
function path(singleton: string, ...steps: [string, string, Record<string, unknown>][]): unknown {
  let nested: unknown = { target: {} };
  for (const [i, o, keys] of [...steps].reverse()) nested = { entry: { i, o, properties: keys, next: nested } };
  return { singleton, next: nested };
}

/** The frames of a map whose symbols are each given a reference: within the transaction that runs it, each symbol binds
 * what its reference designates, so that the map gives one frame, or none where `held` holds nothing one designates
 * ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md)). */
function bound(held: any, references: Record<string, unknown>): any[] {
  return held.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols(references)]))]));
}

/** What an object's reference designates in `held`, by a map whose one symbol is given it; null where it holds none. */
function designated(held: any, reference: unknown): unknown {
  const frames = bound(held, { x: reference });
  return frames.length === 0 ? null : frames[0].get("x");
}

/** The frames of a map whose symbols are given the references a frame's reference binds, read off its plain form, as a
 * program it is handed to reads them: the frame again, or none. */
function framed(held: any, reference: any): any[] {
  const OfObject = held.registered("References.OfObject");
  const binds = (Plain.ToPlain(held)(reference.as_schema(), reference) as Map<string, any>).get("binds") ?? new Map();
  return bound(held, Object.fromEntries([...binds].map(([name, given]: [string, unknown]) =>
    [name, Plain.FromPlain(held)(OfObject, given)])));
}

<a id="REF-01"></a>
## REF-01 · Every store holds the reference schemas, its own; References.install, run again, writes nothing

In [ ]:
const OfObject = store.registered("References.OfObject");
const [OfFrame, OfEntry] = [store.registered("References.OfFrame"), store.registered("References.OfEntry")];
assert(equal([...OfObject.properties.keys()], ["path"]) && OfObject.properties.get("path").schema === store.registered("Paths.OfRoot"));
const OfProperties = store.registered("References.OfProperties");
assert(equal([...OfFrame.properties.keys()], ["binds", "variables", "assigns", "entries", "errors"]));
assert(OfFrame.properties.get("binds").schema.element === OfObject && OfFrame.properties.get("assigns").schema.element === OfProperties);
assert(OfFrame.properties.get("entries").schema.element === OfEntry && equal([...OfProperties.properties.keys()], ["object", "properties"]));
assert(equal([...OfEntry.properties.keys()], ["relation", "links", "properties"]));
assert(OfEntry.properties.get("relation").schema === store.registered("Paths.OfRoot"));
assert(OfEntry.properties.get("links").schema.element === OfObject);
const OfError = store.registered("References.OfError");
assert(OfFrame.properties.get("errors").schema.element === OfError && equal([...OfError.properties.keys()], ["subterm", "kind", "message"]));
assert(![OfObject, OfFrame, OfProperties, OfEntry, OfError].some((schema) => schema.ref)); // value objects, as a path is
const other: any = new B.Store();
assert(other.registered("References.OfObject") !== OfObject); // each store's its own
const names = new Map((store.names() as string[]).map((name) => [name, store.registered(name)]));
store.transaction(References.install); // each found by its name
assert(store.names().length === names.size && (store.names() as string[]).every((name) => store.registered(name) === names.get(name)));

<a id="REF-02"></a>
## REF-02 · An object's reference by its path: a singleton, an owned object, a schema object; each designated by a map whose symbol is given it

In [ ]:
{
  assert(equal(plain(References.of(store, book)), map({ path: path("crm.TheBook") })));
  assert(equal(plain(References.of(store, ada)), map({ path: path("crm.TheBook", ["contacts", "book", { key: "ada" }]) })));
  assert(equal(plain(References.of(store, Person)), map({ path: path(".", ["schemas", "", { name: "crm" }],
    ["schemas", "", { name: "Person" }]) })));
  for (const target of [book, ada]) { // by a map whose symbol is given it: what it designates, by its reference again
    assert(equal(plain(designated(store, References.of(store, target))), plain(target)));
  }
  assert(designated(store, References.of(store, Person)) === Person); // a schema object: itself
  const built = store.References.OfObject((o: any) => o.path((p: any) => p.singleton("crm.TheBook").next((n: any) => n.entry(
    (e: any) => e.i("contacts").o("book").properties({ key: "bob" }).next((n: any) => n.target((t: any) => t))))));
  assert(equal(plain(designated(store, built)), plain(bob))); // written through the builders' fluent calls
}

<a id="REF-03"></a>
## REF-03 · A counted object has no reference, nor a frame that binds one: an upsert's, which the executor alone finds again

In [ ]:
const made = store.transaction(notes);
assert(equal(plains(made), [ada, bob, cy].map((x) => new Map([["contact", plain(x)], ["note", "counted"]]))));
raises(LookupError, () => made[0].get("note"), "a counted 'crm.Note' has no reference: nothing names it"); // the frame given back refuses it
raises(LookupError, () => References.of(store, made[0]), "symbol 'note' of the frame: a counted 'crm.Note' has no reference");
raises(LookupError, () => store.transaction(() => { // read within one: none either
  References.of(store, store.crm.Note((n: any) => n.text("hi").about([contact("ada")])));
}), "a counted 'crm.Note' has no reference: nothing names it");
const again = store.transaction(notes); // run again: the executor finds what it made, which nothing outside it names
assert(equal(plains(again), plains(made)) && again[0].assigns.size === 0); // its notes refused alike: nothing names them

<a id="REF-04"></a>
## REF-04 · A relation entry by its relation's path and all its links and properties: a keyed relation's, one without properties; refused where it links a counted object

In [ ]:
{
  store.transaction(() => {
    const held = [...store.singleton("crm.TheBook").contacts][0]; // crm.Holds, keyed by `key`
    assert(equal(plain(References.of(store, held)), map({
      relation: path(".", ["schemas", "", { name: "crm" }], ["schemas", "", { name: "Holds" }]),
      links: { book: { path: path("crm.TheBook") },
        contact: { path: path("crm.TheBook", ["contacts", "book", { key: "ada" }]) } },
      properties: { key: { str: "ada" } } })));
    const knows = [...contact("ada").knows];
    assert(equal(plain(References.of(store, knows[1])).get("properties"), map({ since: { int: 2021n } })));
  });
  store.transaction(() => {
    store.crm.Person(contact("bob"), (b: any) => b.knows((a: any) => a.insert((e: any) => e.known(contact("cy"))))); // since left unset
  });
  store.transaction(() => {
    const bare = [...contact("bob").knows][0];
    assert(equal(plain(References.of(store, bare)).get("properties"), new Map()));
  });
  raises(LookupError, () => store.transaction(() => { // crm.About links a note
    const note = store.crm.Note((n: any) => n.text("hi").about([contact("ada")])); // a note no frame gives back: made here
    References.of(store, [...note.about][0]);
  }), "a counted 'crm.Note' has no reference");
  /** The frames of a map over the references of `x` and `y`, whose spec is the crm.Knows entry linking them, with
   * `since` where given: an entry's reference is matched by a pattern over the objects it links, one frame where the
   * entry holds, none otherwise. */
  const known = (x: unknown, y: unknown, ...since: bigint[]): any[] => store.transaction((t: any) => t.mutations([
    (mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
      .symbols({ x: References.of(store, x), y: References.of(store, y) })
      .spec([(_: any) => _.x.knows([(e: any) => since.length > 0 ? e.known(_.y).since(...since) : e.known(_.y)])])]))]));
  assert(equal(plains(known(ada, bob, 2020n)), [new Map([["x", plain(ada)], ["y", plain(bob)]])]));
  assert(equal(plains(known(ada, cy, 2021n)), [new Map([["x", plain(ada)], ["y", plain(cy)]])]));
  assert(equal(plains(known(bob, cy)), [new Map([["x", plain(bob)], ["y", plain(cy)]])])); // since unset: the entry with its links alone
  assert(known(ada, cy, 1999n).length === 0 && known(cy, ada).length === 0); // a property no entry has; links no entry has
  const held = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ b: References.of(store, book), c: References.of(store, ada) })
    .spec([(_: any) => _.b.contacts([(e: any) => e.contact(_.c).key("ada")])])]))]));
  assert(equal(plains(held), [new Map([["b", plain(book)], ["c", plain(ada)]])])); // held
}

<a id="REF-05"></a>
## REF-05 · A frame a transaction gave back: each symbol's object; a map whose symbols are given its references gives the frame again

In [ ]:
const frame = frames[0];
{
  const reference = References.of(store, frame);
  assert(equal([...plain(reference).get("binds").keys()].sort(), [...frame.keys()].sort()));
  assert(equal(plain(reference).get("binds").get("ada"), plain(References.of(store, ada))));
  assert(equal(plain(reference).get("assigns").map((a: any) => a.get("properties")),
    ["Ada", "Bob", "Cy"].map((n) => map({ name: { str: n } })))); // what it witnesses
  assert(equal(plains(framed(store, reference)), plains([frame]))); // a map whose symbols are given the references it binds: the frame again
  assert(equal(plain(References.of(store, new Map())), map({ binds: {} }))
    && equal(framed(store, References.of(store, new Map())), [new Map()]));
  assert(equal(framed(store, store.References.OfFrame((f: any) => f)), [new Map()])); // no binds: the empty frame
}

<a id="REF-06"></a>
## REF-06 · What is refused: a counted object nothing names, another store's object, what is not an object, an entry or a frame

In [ ]:
{
  store.transaction(() => {
    const loose = store.crm.Note((n: any) => n.text("by hand")); // counted, written outright: nothing names it
    raises(LookupError, () => References.of(store, loose), "a counted 'crm.Note' has no reference: nothing names it");
    raises(LookupError, () => References.of(store, new Map([["loose", loose]])),
      "symbol 'loose' of the frame: a counted 'crm.Note' has no reference");
    raises(TypeError, () => References.of(other, contact("ada")), "the object belongs to another store");
  });
  raises(TypeError, () => References.of(store, "x"), "References.of takes an object, a relation entry or a frame, not 'x'");
  raises(TypeError, () => References.of(store, new Map([["x", "x"]])),
    "References.of takes an object, a relation entry or a frame, not 'x'"); // a frame binding what is not an object
  store.transaction(() => {
    store.crm.Person(contact("ada"), (a: any) => a.trio((t: any) => t.insert((e: any) => e.b(contact("bob")).c(contact("cy")))));
  });
  store.transaction(() => { // an entry of a relation with no name of its own
    raises(LookupError, () => References.of(store, [...contact("ada").trio][0]), "has no path");
  });
}

<a id="REF-07"></a>
## REF-07 · A reference written and read back as a value; no frame where the store holds nothing it designates

In [ ]:
{
  const held = store.transaction(() => // as the store holds it now: an update of ada's rewrote her entries
    References.of(store, [...store.singleton("crm.TheBook").contacts][0])); // its reference, taken where read
  const backs = new Map<string, any>();
  for (const [name, reference] of [["ada", References.of(store, ada)], ["held", held], ["frame", References.of(store, frame)]] as [string, any][]) {
    const text = Json.ToJSON(store)(reference.as_schema(), reference);
    backs.set(name, Json.FromJSON(store)(reference.as_schema(), text));
    assert(equal(plain(backs.get(name)), plain(reference))); // read back the same
  }
  assert(equal(plain(designated(store, backs.get("ada"))), plain(ada)) && equal(plains(framed(store, backs.get("frame"))), plains([frame])));
  const empty: any = new B.Store(); // its schemas and nothing else
  empty.transaction(crm);
  for (const target of [ada, Person]) {
    const reference = References.of(store, target);
    const back = Json.FromJSON(empty)(empty.registered(store.name_of(reference.as_schema())),
      Json.ToJSON(store)(reference.as_schema(), reference));
    assert(designated(empty, back) === (target === Person ? empty.registered("crm.Person") : null));
  }
  const nowhere = store.References.OfObject((o: any) => o.path((p: any) => p.singleton(".").next((n: any) => n.entry(
    (e: any) => e.i("schemas").o("x").properties({ name: "crm" }).next((n: any) => n.target((t: any) => t))))));
  assert(designated(store, nowhere) === null); // a schema object's path whose steps name no entries
}

<a id="REF-08"></a>
## REF-08 · A frame as a witness: property values an object holds, and relation entries, each by its objects' references

In [ ]:
const [atAda, atBob, knowing] = store.transaction(() => [ // their paths and an entry's reference, taken where they are read
  Paths.of(contact("ada")), Paths.of(contact("bob")), plain(References.of(store, [...contact("ada").knows][0]))]);
{
  const witness = store.References.OfFrame((f: any) => f
    .binds({ ada: (o: any) => o.path(atAda), bob: (o: any) => o.path(atBob) })
    .assigns([(x: any) => x.as_object((p: any) => p.object((o: any) => o.path(atAda)).properties({ name: "Ada" }))])
    .entries([(x: any) => x.as_object((e: any) => e.relation(Paths.of(store.registered("crm.Knows"))).links({
      knower: (o: any) => o.path(atAda), known: (o: any) => o.path(atBob) }).properties({ since: 2020n }))]));
  assert(equal(plain(witness).get("assigns"), [map({ object: plain(References.of(store, ada)), properties: { name: { str: "Ada" } } })]));
  assert(equal(plain(witness).get("entries"), [knowing])); // an OfEntry, as of() gives one
  const text = Json.ToJSON(store)(witness.as_schema(), witness);
  assert(equal(plain(Json.FromJSON(store)(witness.as_schema(), text)), plain(witness))); // carried, and read back the same
  assert(equal(plains(framed(store, witness)), [new Map([["ada", plain(ada)], ["bob", plain(bob)]])])); // its bindings, by a map over their references
}

<a id="REF-09"></a>
## REF-09 · A pattern's symbol given a reference in place of a schema: it binds what the reference designates, one frame, or none; its spec filters or is written as any symbol's, by the schema its path implies where the store holds nothing it designates; a symbol has exactly one of a schema and a reference

In [ ]:
{
  const reference = References.of(store, ada); // a symbol given a reference binds what it designates, one frame each
  const known = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: reference, y: Person }).spec([(_: any) => _.x.knows([_.y])])]))]));
  assert(equal(plains(known), [new Map([["x", plain(ada)], ["y", plain(bob)]]), new Map([["x", plain(ada)], ["y", plain(cy)]])])); // along an adjacency
  assert(equal(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: References.of(store, bob), y: Person }).spec([(_: any) => _.x.knows([_.y])])]))]))
    .map((f: any) => plain(f.get("y"))), [plain(cy)]));
  assert(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: reference }).spec([(_: any) => _.x.name("Bob")])]))])).length === 0); // its spec filters, as any symbol's
  const upserted = store.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ x: References.of(store, cy) }).spec([(_: any) => _.x.name("Cy C.")])]))]));
  assert(equal(plains(upserted), [new Map([["x", plain(cy)]])]) && equal(upserted[0].assigns, map({ x: { name: "Cy C." } }))); // an upsert writes its spec over what it binds
  assert(equal(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ s: References.of(store, Person) }).spec([(_: any) => _.s.name("crm.Person")])]))])), [new Map([["s", Person]])]));
  const nowhere = store.References.OfObject((o: any) => o.path((p: any) => p.singleton("crm.TheBook").next((n: any) => n.entry(
    (e: any) => e.i("contacts").o("book").properties({ key: "zed" }).next((n: any) => n.target((t: any) => t))))));
  for (const term of [(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: nowhere })])),
    (mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({ x: nowhere })]))]) {
    assert(store.transaction((t: any) => t.mutations([term])).length === 0); // it designates nothing: no frame, nothing made
  }
  assert(store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: nowhere }).spec([(_: any) => _.x.name("Zed")])]))])).length === 0); // its spec, by the schema its path implies

  /** A `References.OfObject` of `held`'s, from a singleton through entries, each [i, o, keys]. */
  const at = (held: any, singleton: string, ...steps: [string, string, Record<string, unknown>][]): any => {
    const nested = (n: any, rest: [string, string, Record<string, unknown>][]): any => {
      if (rest.length === 0) return n.target((t: any) => t);
      const [[i, o, keys], ...more] = rest as [[string, string, Record<string, unknown>], ...[string, string, Record<string, unknown>][]];
      return n.entry((e: any) => e.i(i).o(o).properties(keys).next((n: any) => nested(n, more)));
    };
    return held.References.OfObject((o: any) => o.path((p: any) => p.singleton(singleton).next((n: any) => nested(n, steps))));
  };

  const client: any = new B.Store(); // the schemas, not the data: its spec is written by the schemas its reference's path goes through
  client.transaction(crm);
  const sent = client.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: at(client, "crm.TheBook", ["contacts", "book", { key: "ada" }]), y: client.registered("crm.Person") })
    .spec([(_: any) => _.x.knows([_.y])])]))]));
  const text = Json.dumps(Plain.ToPlain(client).Transaction(sent)); // written as JSON, read where the data is, and run
  assert(equal(plains(store.transaction(Plain.FromPlain(store).Transaction(Json.loads(text)))), plains(known))); // the frames, as given above

  /** crm.Firm, owned through crm.Holds as a person is, by an adjacency also named `book`. */
  const firm = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("firm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Holds: S.OfRelation, Firm: S.OfObject }).spec([
        (_: any) => _.Holds.name("crm.Holds"),
        (_: any) => _.Firm.name("crm.Firm").ref(true).adjacencies([
          (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true)]),
      ])]))]);
  };

  const firms: any = new B.Store();
  firms.transaction(crm);
  firms.transaction(firm);
  for (const [held, given, why] of [
    [client, at(client, "crm.Nope"), "starts from 'crm.Nope', which no schema declares a singleton"],
    [client, at(client, "crm.TheBook", ["nope", "book", {}]),
      "steps through 'nope', an adjacency 'crm.Book' does not declare"],
    [client, at(client, "crm.TheBook", ["contacts", "nope", {}]), "steps through 'contacts' of 'crm.Book', where not "
      + "exactly one schema declares an adjacency 'nope' marked owner() on another link of its relation: found []"],
    [firms, at(firms, "crm.TheBook", ["contacts", "book", { key: "ada" }]), "steps through 'contacts' of 'crm.Book', "
      + "where not exactly one schema declares an adjacency 'book' marked owner() on another link of its relation: "
      + "found ['crm.Firm', 'crm.Person']"],
    [client, at(client, ".", ["schemas", "", { name: "Nope" }]), "names a schema object the store does not hold"],
  ] as [any, any, string][]) {
    const message = `symbol 'x' is given a reference whose path ${why}: a spec writes what its object's schema declares`;
    raises(LookupError, () => held.transaction(() => held.Schemas.OfPattern((p: any) => p.symbols({ x: given })
      .spec([(_: any) => _.x.name("Zed")]))), message);
  }
  let both: any, neither: any, given: any;
  store.transaction(() => {
    both = store.Schemas.OfPattern((p: any) => p.symbols([(s: any) => s.name("x").schema([Person]).reference(reference)]));
    neither = store.Schemas.OfPattern((p: any) => p.symbols([(s: any) => s.name("x")]));
    given = store.Schemas.OfPattern((p: any) => p.symbols({ x: reference }));
  });
  assert(equal(References.validate(both), ["symbol 'x' has both a schema and a reference: a symbol has exactly one"]));
  assert(equal(References.validate(neither), ["symbol 'x' has neither a schema nor a reference: a symbol has exactly one"]));
  assert(References.validate(given).length === 0 && given.symbols.get("x").schema === null);
  raises(TypeError, () => References.validate("x"), "References.validate takes a pattern, not 'x'");
  raises(TypeError, () => store.transaction(() => store.Schemas.OfPattern((p: any) => p.symbols([(s: any) => s.name("x")
    .schema([Person]).reference(reference)]).spec([(_: any) => _.x.name("Ada")]))),
    "symbol 'x' has both a schema and a reference: a symbol has exactly one");
}

<a id="REF-10"></a>
## REF-10 · A frame annotated with errors: its reference carries each error's subterm, kind and message, written and read back as any value is

In [ ]:
{
  store.transaction(() => {
    store.crm.Book(store.singleton("crm.TheBook"), (b: any) => b.contacts({ eve: (e: any) => e.name("Eve") }));
  });
  const wire = store.transaction(() => { // written where Eve is read: Eve by her path
    const eve = contact("eve");
    return Plain.ToPlain(store).Transaction(store.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m
      .symbols([(p: any) => p.symbols({ x: Person }).spec([(_: any) => _.x.name("Ada")])]).then([(w: any) => w.update((u: any) => u
      .apply([(p: any) => p.symbols({ x: Person }).spec([(_: any) => _.x.knows([eve])])]))]))])));
  });
  const knowing = Plain.FromPlain(store).Transaction(wire);
  store.transaction(() => {
    store.crm.Book(store.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("eve")))); // Eve deleted
  });
  let annotated: any = null;
  try {
    store.transaction(knowing);
  } catch (error) {
    if (!(error instanceof LookupError)) throw error;
    [annotated] = (error as any).frames;
  }
  assert(equal(plains([annotated]), [new Map([["x", plain(ada)]])]) && annotated.errors.length === 1);
  const reference = References.of(store, annotated);
  assert(equal(plain(reference), map({ binds: { x: plain(References.of(store, ada)) }, assigns: [
    { object: plain(References.of(store, ada)), properties: { name: { str: "Ada" } } }], errors: [{
    subterm: "$.mutations[0].map.then[0].update", kind: "LookupError",
    message: "no object of the store has the path a spec refers to" }] })));
  const text = Json.ToJSON(store)(reference.as_schema(), reference);
  assert(equal(plain(Json.FromJSON(store)(reference.as_schema(), text)), plain(reference))); // carried, the error itself not
  assert(!plain(References.of(store, frame)).has("errors")); // a frame annotated with none: written as before
  const built = store.References.OfFrame((f: any) => f.binds({ x: (o: any) => o.path(atAda) }).errors([
    (x: any) => x.as_object((e: any) => e.subterm("$.mutations[0].upsert").kind("ValueError").message("refused"))]));
  assert(equal(plain(built).get("errors"), [map({ subterm: "$.mutations[0].upsert", kind: "ValueError", message: "refused" })]));
}

<a id="REF-11"></a>
## REF-11 · A parameter reference holds its parameter's name and nothing else: a value of the store's, written and read back, which no store resolves

In [ ]:
{
  const OfParameter = store.registered("References.OfParameter");
  assert(equal([...OfParameter.properties.keys()], ["name"]) && OfParameter.properties.get("name").schema === store.registered("Schemas.OfStr"));
  assert(!OfParameter.ref); // a value object, as the other references are
  const n = store.References.OfParameter((p: any) => p.name("n")); // built by its name: References.of gives none
  assert(equal(plain(n), map({ name: "n" })));
  const text = Json.ToJSON(store)(OfParameter, n);
  assert(text === '{"name": "n"}' && equal(plain(Json.FromJSON(store)(OfParameter, text)), map({ name: "n" })));
  raises(TypeError, () => References.of(store, "n"), // a parameter's name designates no object a store holds
    "References.of takes an object, a relation entry or a frame, not 'n'");
}

<a id="REF-12"></a>
## REF-12 · A parameter reference stands in the place of anything with a schema: a property's scalar or value object, a container's element, a union's branch, an intersection's part and an entry's property; given as the value, or built through `as_parameter`; read through `as_parameter`, and as empty by every other visitor

In [ ]:
/** par.Point, a value object; par.Shape, a union of scalars; par.Spot, a union of points; par.Series, a union of
 * containers; par.Box, a singleton holding a value of each kind, and near boxes, at a gap. */
function par(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Point: S.OfObject, Shape: S.OfUnion, Spot: S.OfUnion, Series: S.OfUnion, Near: S.OfRelation,
      Box: S.OfObject })
    .spec([
      (_: any) => _.Point.name("par.Point").properties({ x: S.OfInt, y: S.OfInt }),
      (_: any) => _.Shape.name("par.Shape").branches([
        (t: any) => t.name("radius").schema([S.OfFloat]), (t: any) => t.name("side").schema([S.OfInt])]),
      (_: any) => _.Spot.name("par.Spot").branches([
        (t: any) => t.name("corner").schema([_.Point]), (t: any) => t.name("centre").schema([_.Point])]),
      (_: any) => _.Series.name("par.Series").branches([
        (t: any) => t.name("ints").schema([S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt]))]),
        (t: any) => t.name("strs").schema([S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr]))])]),
      (_: any) => _.Near.name("par.Near").links([(t: any) => t.name("box"), (t: any) => t.name("other")]).properties({
        gap: S.OfInt }),
      (_: any) => _.Box.name("par.Box").ref(true).singleton("par.TheBox").properties({
        size: S.OfInt, at: _.Point, shape: _.Shape, spot: _.Spot, series: _.Series,
        sizes: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt])),
        points: S.OfContainer((c: any) => c.index([S.OfStr]).element([_.Point])),
        pair: S.OfIntersection((i: any) => i.parts([
          (t: any) => t.name("a").schema([_.Point]), (t: any) => t.name("b").schema([_.Point])])),
        some: S.OfAny((a: any) => a.schemas([S.OfInt, _.Point])),
        any: S.OfAny((a: any) => a),
        param: txn.References.OfParameter,
      }).adjacencies([(t: any) => t.name("near").relation([_.Near]).me("box"), (t: any) => t.relation([_.Near]).me("other")]),
    ])]))]);
}

const held: any = new B.Store();
held.transaction(par);
const Box = held.registered("par.Box");
const n = held.References.OfParameter((p: any) => p.name("n")); // a value object, built by its parameter's name

/** The name of the parameter a `Visitors.OfAny` holds a reference to, or null. */
function parameter(read: any): unknown {
  const found: unknown[] = [];
  read.as_parameter((p: any) => {
    found.push(p.name.get());
    return p;
  });
  return found[0];
}

/** A property's `Visitors.OfAny`. */
function of(prop: any): any {
  const found: any[] = [];
  prop.as_value((a: any) => {
    found.push(a);
    return a;
  });
  return found[0];
}

/** What every other kind's visitor reads of a `Visitors.OfAny`: nothing, where a parameter reference stands in. */
function kinds(read: any): unknown[] {
  const found: unknown[] = [];
  read.as_scalar((s: any) => {
    if (s.get() !== null) found.push(s.get());
    return s;
  });
  for (const select of ["as_object", "as_union", "as_intersection", "as_container"]) {
    read[select]((v: any) => {
      if (v.as_schema() !== null) found.push(v);
      return v;
    });
  }
  return found;
}

held.transaction(() => { // given as the value, in place of anything that has a schema
  const box = held.singleton("par.TheBox");
  held.par.Box(box, (b: any) => b.size(n).at(n).shape((s: any) => s.radius(n))
    .spot((s: any) => s.corner(n)).series((s: any) => s.ints(n))
    .sizes([1n, n, 3n]).points({ o: n }).pair((p: any) => p.a((a: any) => a.x(1n)).b(n))
    .some(n).any(n).param(n).near([(e: any) => e.other(box).gap(n)]));
});
held.transaction(() => {
  const box = held.singleton("par.TheBox");
  const [size, at, shape, spot, series, sizes, points, pair, some] = ["size", "at", "shape", "spot", "series", "sizes",
    "points", "pair", "some"].map((name) => of(box[name]));
  assert(equal([size, at, some].map(parameter), ["n", "n", "n"]) && [size, at, some].every((a) => kinds(a).length === 0));
  assert(box.size.get() === null && box.at.x.get() === null); // read as an unset value is
  assert([shape, spot, series, sizes, points, pair].every((a) => parameter(a) === null)); // each holds a value
  const members: any[] = [];
  shape.as_union((u: any) => members.push(u.radius));
  spot.as_union((u: any) => members.push(u.corner));
  series.as_union((u: any) => members.push(u.ints));
  pair.as_intersection((i: any) => members.push(i.b));
  const [radius, corner, ints, b] = members;
  assert(radius.get() === null && corner.as_schema() === null && ints.length === 0 && b.x.get() === null); // empty
  assert(equal(members.map(parameter), ["n", "n", "n", "n"])); // a branch's or a part's empty visitor holds it
  sizes.as_container((c: any) => members.push(c));
  points.as_container((c: any) => members.push(c));
  const [heldSizes, heldPoints] = members.slice(-2);
  assert(equal([0n, 1n, 2n].map((i) => parameter(heldSizes.at(i))), [null, "n", null]) && kinds(heldSizes.at(1n)).length === 0);
  assert(parameter(heldPoints.at("o")) === "n" && kinds(heldPoints.at("o")).length === 0);
  assert(box.near[0].gap.get() === null && parameter(of(box.near[0].gap)) === "n"); // an entry's property
  for (const admitted of [of(box.any), of(box.param)]) { // where its own schema is admitted, a parameter reference is a value of it, too
    assert(parameter(admitted) === "n" && equal(kinds(admitted).map((o: any) => o.name.get()), ["n"]));
  }
});
held.transaction(() => { // built through as_parameter, where a callback writes a value
  const box = held.singleton("par.TheBox");
  held.par.Box(box, (b: any) => b.size((s: any) => s.as_value((a: any) => a.as_parameter((p: any) => p.name("m"))))
    .at((s: any) => s.as_value((a: any) => a.as_parameter((p: any) => p.name("m")).as_parameter((p: any) => p.name("k")))) // the same builder
    .sizes((s: any) => s.as_value((a: any) => a.as_container((c: any) => c.at(0n).as_parameter(n))))
    .some((s: any) => s.as_value((a: any) => a.as_parameter(box.size)))); // a property's value, as held before
});
held.transaction(() => {
  const box = held.singleton("par.TheBox");
  assert(equal(["size", "at", "some"].map((name) => parameter(of(box[name]))), ["m", "k", "n"]));
  const found: unknown[] = [];
  of(box.sizes).as_container((c: any) => found.push(parameter(c.at(0n))));
  assert(found[0] === "n");
  held.par.Box(box, (b: any) => b.size((s: any) => s.as_value((a: any) => a.as_parameter((p: any) => p.name("j"))))); // edited
  assert(parameter(of(box.size)) === "j");
  held.par.Box(box, (b: any) => b.size(7n).at((a: any) => a.x(2n)).spot((s: any) => s.corner((c: any) => c.x(3n))));
  assert(box.size.get() === 7n && parameter(of(box.size)) === null && box.at.x.get() === 2n); // a value written over it
  assert(parameter(of(box.at)) === null);
  found.push(box.spot.corner.x.get());
  assert(found[1] === 3n);
});

<a id="REF-13"></a>
## REF-13 · A parameter reference in the place of another schema's value is written marked, `{"$schema": "References.OfParameter", "name": n}`, and read back the same: in documents, snapshots, JSON, YAML, standalone values and a spec's payload; bare where its own schema is declared

In [ ]:
{
  held.transaction(() => {
    const box = held.singleton("par.TheBox");
    held.par.Box(box, (b: any) => b.size(n).at(n).spot((s: any) => s.corner(n)));
  });
  const marked = map({ $schema: "References.OfParameter", name: "n" });
  const document = held.transaction(() => Plain.ToPlain(held).OfObject(Box, held.singleton("par.TheBox")));
  const written = new Map<string, any>();
  for (const item of document.get("mutations")[0].get("upsert").get("exists")[0].get("spec")[0].get("0")) {
    if (item.has("as_property")) {
      written.set(item.get("as_property").get("name"), item.get("as_property").get("mutations")[0].get("set").get("value"));
    }
  }
  assert(["size", "at", "some"].every((name) => equal(written.get(name), marked))); // in another schema's place: marked
  assert(equal(written.get("shape"), map({ radius: marked })) && equal(written.get("spot"), map({ corner: marked })));
  assert(equal(written.get("series"), map({ ints: marked })) && equal(written.get("pair"), map({ a: { x: 1n }, b: marked })));
  assert(equal(written.get("sizes"), [marked, marked, 3n]) && equal(written.get("points"), map({ o: marked })));
  assert(equal(written.get("param"), map({ name: "n" }))); // where its own schema is declared: bare
  assert(equal(written.get("any"), map({ "References.OfParameter": { name: "n" } }))); // of any schema: tagged by its schema, as any value is
  const again: any = new B.Store();
  again.transaction(par);
  const snapshot = held.transaction(() => Json.loads(Json.dumps(Plain.ToPlain(held).Snapshot())));
  Plain.FromPlain(again).Snapshot(snapshot); // read back the same, in a snapshot
  again.transaction(() => {
    assert(equal(Plain.ToPlain(again).OfObject(again.registered("par.Box"), again.singleton("par.TheBox")), document));
  });
  held.transaction(() => {
    for (const text of [Json.ToJSON(held)(Box, held.singleton("par.TheBox")), YAML.ToYAML(held)(Box, held.singleton("par.TheBox"))]) {
      const reader = text.startsWith("{") ? Json.FromJSON(again) : YAML.FromYAML(again);
      again.transaction(() => {
        assert(equal(Plain.ToPlain(again).OfObject(again.registered("par.Box"), reader(again.registered("par.Box"), text)), document));
      });
    }
  });
  const Point = held.registered("par.Point");
  const point = held.par.Point((p: any) => p.x(n).y(2n)); // a standalone value
  assert(equal(Plain.ToPlain(held)(Point, point), map({ x: marked, y: 2n })));
  assert(equal(Plain.ToPlain(held)(Point, Plain.FromPlain(held)(Point, map({ x: marked, y: 2n }))), map({ x: marked, y: 2n })));
  const OfParameter = held.registered("References.OfParameter");
  assert(equal(Plain.ToPlain(held)(OfParameter, Plain.FromPlain(held)(OfParameter, marked)), map({ name: "n" }))); // its own schema's
  const sent = held.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ x: Box }).spec([(_: any) => _.x.size(held.References.OfParameter((p: any) => p.name("w")))])]))]));
  const wire = Plain.ToPlain(held).Transaction(sent); // a spec's payload, marked as a property's value is
  assert(equal(wire.get("mutations")[0].get("upsert").get("exists")[0].get("spec")[0].get("x"), map([{ as_property: { name: "size", mutations: [
    { set: { value: { $schema: "References.OfParameter", name: "w" } } }] } }])));
  held.transaction(Plain.FromPlain(held).Transaction(Json.loads(Json.dumps(wire))));
  held.transaction(() => {
    assert(parameter(of(held.singleton("par.TheBox").size)) === "w");
  });
}

<a id="REF-14"></a>
## REF-14 · Validation accepts a parameter reference in any place: it is never a wrong value, and it holds an intersection's part; where its parameter context maps its name to nothing, it is partial, and valid

In [ ]:
{
  held.transaction(() => {
    const box = held.singleton("par.TheBox");
    held.par.Box(box, (b: any) => b.size(n).at(n).shape((s: any) => s.radius(n))
      .spot((s: any) => s.corner(n)).sizes([n]).points({ o: n }).some(n)
      .pair((p: any) => p.a((a: any) => a.x(1n)).b((q: any) => q.x(2n)))
      .near([(e: any) => e.other(box).gap(n)]));
    assert(equal(Validators.Validate(held)(Box, box), [])); // never a wrong value, in any place
    const outcome = Validators.Check(held)(Box, box);
    assert(equal(outcome.problems, []) && equal(outcome.unknowns, []) && outcome.holds === true);
    assert(equal(outcome.partials, ["size", "at", "shape.radius", "spot.corner", "series.ints", "sizes[0]", "points['o']",
      "some", "near[0].gap", "near[1].gap"].map((place) => `par.Box#0.${place}: parameter 'n' is neither assigned nor bound in its context`))
      && outcome.result === "partial"); // an empty context maps no name: partial, and valid
    held.par.Box(box, (b: any) => b.pair((p: any) => p.a(n).b(n)));
    assert(equal(Validators.Validate(held)(Box, box), [])); // an intersection's part that holds one is held
    const point = held.par.Point((p: any) => p.x(1n)); // a value object assigned: bound where a point is expected
    assert(equal(Validators.Validate(held)(Box, box, new Map([["n", point]])),
      ["size", "shape.radius", "series.ints", "sizes[0]", "near[0].gap", "near[1].gap"].map((place) =>
        `par.Box#0.${place}: parameter 'n' is assigned a value of another schema than its place expects`)));
  });
}

<a id="REF-15"></a>
## REF-15 · A place that holds no parameter reference reads as empty through `as_parameter`; a read-only visitor refuses a write, a builder refuses a value that is not a parameter reference, and a schema object's value holds none

In [ ]:
{
  held.transaction(() => {
    const box = held.singleton("par.TheBox");
    held.par.Box(box, (b: any) => b.size(3n).shape((s: any) => s.radius(n)));
  });
  held.transaction(() => {
    const box = held.singleton("par.TheBox");
    assert(parameter(of(box.size)) === null); // a value of its own schema: no parameter reference held
    raises(ReadOnlyError, () => of(box.size).as_parameter(n), // read-only: refused, saying why
      "property 'size' holds no parameter reference");
    raises(ReadOnlyError, () => of(box.at).as_parameter(n), "property 'at' is read-only; write through a builder");
    raises(ReadOnlyError, () => box.shape.radius.as_parameter(n), "branch 'radius' holds a parameter reference");
    raises(TypeError, () => held.par.Box(box, (b: any) => b.size((s: any) => s.as_value((a: any) => a.as_parameter(4n)))),
      "property 'size' takes a parameter reference or a Spec, got int");
  });
  const scalar = new B.OfScalar.Builder(null, true, held).name("par.Name"); // a schema object's value holds none
  const reasons: unknown[] = [];
  scalar.name.as_value((v: any) => v.as_parameter((p: any) => {
    reasons.push(p.as_schema());
    return p;
  }));
  assert(equal(reasons, [null]));
  raises(TypeError, () => scalar.name.as_value((v: any) => v.as_parameter(n)), "property 'name' holds no parameter reference");
}

<a id="REF-16"></a>
## REF-16 · A parameter reference stands in the place of an object a link links: given as the object is, or by the peer shorthand; read through the link as an empty visitor whose `as_parameter` hands it over; holding no entry table, keyed by its parameter's name, never reached, accepted by validation, halting comparison, removed by its name, and refused by a relation that owns what it links

In [ ]:
/** lnk.Hub, a singleton owning its nodes by key; lnk.Node, each linked to the next, at a weight. */
function lnk(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("lnk/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Holds: S.OfRelation, Next: S.OfRelation, Tags: S.OfRelation, Hub: S.OfObject, Node: S.OfObject })
    .spec([
      (_: any) => _.Holds.name("lnk.Holds").links([(t: any) => t.name("hub"), (t: any) => t.name("node")]).properties({
        key: S.OfStr }).uniques([["hub", "key"], ["node"]]),
      (_: any) => _.Next.name("lnk.Next").links([(t: any) => t.name("from"), (t: any) => t.name("to")]).properties({
        w: S.OfInt }).uniques([[]]), // a set
      (_: any) => _.Tags.name("lnk.Tags").links([(t: any) => t.name("tagger"), (t: any) => t.name("tagged"),
        (t: any) => t.name("tag")]),
      (_: any) => _.Hub.name("lnk.Hub").ref(true).singleton("lnk.TheHub").adjacencies({
        nodes: (t: any) => t.relation([_.Holds]).me("hub") }),
      (_: any) => _.Node.name("lnk.Node").ref(true).properties({ label: S.OfStr }).adjacencies([
        (t: any) => t.name("hub").relation([_.Holds]).me("node").owner(true),
        (t: any) => t.name("next").relation([_.Next]).me("from"),
        (t: any) => t.name("prev").relation([_.Next]).me("to"),
        (t: any) => t.name("tags").relation([_.Tags]).me("tagger"),
        (t: any) => t.relation([_.Tags]).me("tagged"), // anonymous
      ]),
    ])]))]);
}

const graph: any = new B.Store();
graph.transaction(lnk);
const [Hub, Node] = ["lnk.Hub", "lnk.Node"].map((name) => graph.registered(name));
const k = graph.References.OfParameter((p: any) => p.name("k"));
graph.transaction(() => { // in the place of the object a link links: given as the object is, or by the peer shorthand
  const hub = graph.singleton("lnk.TheHub");
  graph.lnk.Hub(hub, (h: any) => h.nodes({ a: (e: any) => e.node((x: any) => x.label("A")),
    b: (e: any) => e.node((x: any) => x.label("B")) }));
  const [a, b] = ["a", "b"].map((key) => hub.nodes({ key }));
  graph.lnk.Node(a, (x: any) => x.next([(e: any) => e.to(k).w(1n), (e: any) => e.to(b).w(2n), (e: any) => e.to(k).w(1n)]));
  graph.lnk.Node(b, (x: any) => x.next([k])); // the peer shorthand
});
graph.transaction(() => {
  const hub = graph.singleton("lnk.TheHub");
  const [a, b] = ["a", "b"].map((key) => hub.nodes({ key }));
  assert(a.next.length === 2 && b.next.length === 1); // an equal entry elided: one parameter, one key, by its name
  const entry = a.next[0];
  assert(entry.to.as_schema() === null && entry.to.label.get() === null && [...entry.to.next].length === 0); // empty
  const names: unknown[] = [];
  entry.to.as_parameter((p: any) => names.push(p.name.get()));
  b.next[0].to.as_parameter((p: any) => names.push(p.name.get()));
  assert(equal(names, ["k", "k"]) && a.next[1].to === b && entry.w.get() === 1n);
  assert(equal([...b.prev], [a.next[1]])); // an object linked holds its entry; a parameter reference holds none
  assert(same(Reachable.of(hub), [hub, a, b])); // never reached
  assert(equal(Validators.Validate(graph).Reachable(Hub, hub), [])); // accepted in an object's place: partial, and valid
  assert(Validators.Check(graph).Reachable(Hub, hub).result === "partial");
  const next = Node.adjacencies.get("next");
  assert(new C.OfEntry(next, entry).compare(new C.OfEntry(next, entry)) === null);
  assert(new C.OfEntry(next, a.next[1]).compare(new C.OfEntry(next, a.next[1])) === 0);
  const reference = References.of(graph, entry); // an entry's reference: the parameter reference in the object's place
  assert(equal(Plain.ToPlain(graph)(reference.as_schema(), reference).get("links").get("to"), map({
    $schema: "References.OfParameter", name: "k" })));
  graph.lnk.Node(b, (x: any) => x.next((adj: any) => adj.remove((e: any) => e.to(k)))); // removed by its parameter's name
  assert(b.next.length === 0);
  graph.lnk.Node(b, (x: any) => x.next([k]));
  const built: unknown[] = [];
  graph.lnk.Node(b, (x: any) => x.next((adj: any) => adj.insert((e: any) => built.push(e.to(entry.to).w(9n).to.as_schema()))));
  assert(equal(built, [null]) && b.next[1].to.as_schema() === null); // relinked to what another entry links; read as empty
  graph.lnk.Node(b, (x: any) => x.next(new Map([[4n, k]]))); // a keyed mapping's value: the link takes it
  graph.lnk.Node(b, (x: any) => x.next((adj: any) => adj.remove((e: any) => e.to(a)).insert((e: any) => e.to(k).w(7n))));
  assert(equal([...b.next].map((e: any) => e.w.get()), [null, 9n, 4n, 7n])); // checked against its unique clauses where a removal missed
  raises(TypeError, () => graph.lnk.Node(b, (x: any) => x.next([(t: any) => t.to(k).label("K")])),
    "link 'to' holds a parameter reference, which has no members to write");
  raises(ValueError, () => graph.lnk.Hub(hub, (h: any) => h.nodes({ c: k })), "link 'node' of an entry of 'lnk.Holds' "
    + "holds a parameter reference, which a relation that owns what it links takes in no link");
});

<a id="REF-17"></a>
## REF-17 · A link to a parameter reference is set as `{"reference": {"$schema": "References.OfParameter", "name": n}}`, beside a symbol or a path, and read back the same: in documents, snapshots, a spec and its effect, matched by its parameter's name, and in an expression's nested form; a malformed one refused

In [ ]:
{
  const marked = map({ $schema: "References.OfParameter", name: "k" });
  const [document, atA, snapshot] = graph.transaction(() => {
    const a = graph.singleton("lnk.TheHub").nodes({ key: "a" });
    const written = Plain.ToPlain(graph).OfObject(Node, a);
    const parts = written.get("mutations")[0].get("upsert").get("exists")[0].get("spec").flatMap((item: any) => [...item.values()])
      .flat().filter((member: any) => member.has("as_adjacency")).flatMap((member: any) => member.get("as_adjacency").get("mutations"))
      .flatMap((verb: any) => verb.get("insert").get("entries")).flatMap((e: any) => e.get("mutations"));
    assert(parts.some((part: any) => equal(part, map({ as_link: { name: "to", mutations: [{ set: { reference: marked } }] } }))));
    return [written, References.of(graph, a), Json.loads(Json.dumps(Plain.ToPlain(graph).Snapshot()))];
  }); // written marked beside a symbol or a path
  const copy: any = new B.Store();
  copy.transaction(lnk);
  Plain.FromPlain(copy).Snapshot(snapshot); // read back the same, in a snapshot
  copy.transaction(() => {
    const held = copy.singleton("lnk.TheHub").nodes({ key: "a" });
    const Copied = copy.registered("lnk.Node");
    assert(equal(Plain.ToPlain(copy).OfObject(Copied, held), document));
    const read = Json.FromJSON(copy)(Copied, Json.dumps(document)); // and in a document, read where its object is
    assert(equal(Plain.ToPlain(copy).OfObject(Copied, read), document));
  });
  const seen: any[] = [];
  const end = Changes.observe(graph, (journal: any) => seen.push(Plain.ToPlain(graph)(graph.registered("Changes.Journal"), journal)));
  const spec = graph.Mutations.OfTransaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ x: atA }).spec([
      (_: any) => _.x.next([(e: any) => e.to(graph.References.OfParameter((q: any) => q.name("j"))).w(3n)])])]))]));
  const wire = Json.loads(Json.dumps(Plain.ToPlain(graph).Transaction(spec))); // a spec's link set, marked
  const setJ = map({ as_link: { name: "to", mutations: [{ set: { reference: { $schema: "References.OfParameter", name: "j" } } }] } });
  assert(equal(wire.get("mutations")[0].get("upsert").get("exists")[0].get("spec")[0].get("x")[0].get("as_adjacency")
    .get("mutations")[0].get("insert").get("entries")[0].get("mutations")[0], setJ));
  graph.transaction(Plain.FromPlain(graph).Transaction(wire)); // run: it links the parameter reference
  graph.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ x: atA }).spec([(_: any) => _.x.next(new Map([[5n, graph.References.OfParameter((q: any) => q.name("i"))]]))])]))]));
  graph.transaction(() => { // removed: its journal removes it by its parameter's name
    graph.lnk.Node(graph.singleton("lnk.TheHub").nodes({ key: "a" }), (x: any) => x.next((adj: any) => adj.remove(
      (e: any) => e.to(graph.References.OfParameter((q: any) => q.name("i"))))));
  });
  end();
  assert(equal(seen.flatMap(([transaction]: any) => transaction.get("changes").map((change: any) => change.get("entry").get("verb"))),
    ["insert", "insert", "remove"]));
  assert(equal(seen[0][0].get("changes")[0].get("entry").get("links").get("to"),
    map({ reference: { $schema: "References.OfParameter", name: "j" } })));
  graph.transaction(() => {
    const hub = graph.singleton("lnk.TheHub");
    const held = [...hub.nodes({ key: "a" }).next];
    const names: unknown[] = [];
    for (const entry of held) if (entry.to.as_schema() === null) entry.to.as_parameter((p: any) => names.push(p.name.get()));
    assert(equal(names, ["k", "j"]) && held[1].to === hub.nodes({ key: "b" })); // linked, in its transaction
  });

  /** The frames of a map whose node's spec gives an entry linking a parameter reference to `name`. */
  function linking(name: string): any[] {
    return graph.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
      .symbols({ x: Node }).spec([(_: any) => _.x.next([(e: any) => e.to(graph.References.OfParameter(
        (q: any) => q.name(name)))])])]))]));
  }

  assert(linking("j").length === 1 && linking("k").length === 2 && linking("z").length === 0); // matched by its parameter's name
  raises(DecodeError, () => { // never guessed
    const bad = Json.loads(Json.dumps(wire));
    bad.get("mutations")[0].get("upsert").get("exists")[0].get("spec")[0].get("x")[0].get("as_adjacency").get("mutations")[0]
      .get("insert").get("entries")[0].get("mutations")[0].get("as_link").get("mutations")[0].get("set").get("reference")
      .set("name", 1n);
    Plain.FromPlain(graph).Transaction(bad);
  }, "set.reference.name: expected str, got int");
  const expression = graph.Mutations.OfTransaction((t: any) => t.mutations([k])); // in the place of a term an expression links
  const nested = Plain.ToPlain(graph).Transaction(expression);
  assert(equal(nested, map({ mutations: [marked] })));
  assert(equal(Plain.ToPlain(graph).Transaction(Plain.FromPlain(graph).Transaction(nested)), nested));
  const below: any = new B.Store(); // a store below: an entry another writes there, which no object present above has an
  below.transaction(lnk); // adjacency to by name, is applied to those present directly, a parameter reference as itself
  below.transaction(() => {
    below.lnk.Hub(below.singleton("lnk.TheHub"), (h: any) => h.nodes({ a: (e: any) => e.node((x: any) => x.label("A")) }));
  });
  const above: any = new B.Store(below);
  above.transaction(lnk);
  above.transaction(() => above.singleton("lnk.TheHub").nodes({ key: "a" })); // present above
  below.transaction(() => {
    const hub = below.singleton("lnk.TheHub");
    below.lnk.Hub(hub, (h: any) => h.nodes({ c: (e: any) => e.node((x: any) => x.tags((adj: any) => adj.insert(
      (t: any) => t.tagged(hub.nodes({ key: "a" })).tag(below.References.OfParameter((q: any) => q.name("t")))))) }));
  });
  above.transaction(() => {
    const [tagging] = Proxies._linking(above.registered("lnk.Tags"), "tagged", above.singleton("lnk.TheHub").nodes({ key: "a" }));
    const tag: unknown[] = [];
    tagging.tag.as_parameter((p: any) => tag.push(p.name.get()));
    assert(equal(tag, ["t"]));
  });
}